# N-gram Models in Natural Language Processing (NLP)

Welcome to this interactive notebook! Here, we will explore **N-gram Models**, one of the fundamental concepts in Natural Language Processing (NLP).

## Learning Objectives
By the end of this notebook, you should be able to:
- Understand what an N-gram is and why it is useful in NLP.
- Differentiate between Unigrams, Bigrams, Trigrams, and higher-order N-grams.
- Understand how N-grams preserve word order compared to the Bag-of-Words model.
- Generate N-grams using `scikit-learn`.
- Build feature vectors using `CountVectorizer` with different n-gram ranges.
- Compare feature spaces created by different N-gram models.
- Understand the advantages and limitations of N-gram models.
- Understand the basic idea of an N-gram Language Model for next-word prediction.

---

## 1. Title and Introduction

### What are N-gram models?
An **N-gram** is a contiguous sequence of *N* items from a given sample of text or speech. The items can be phonemes, syllables, letters, or words. In most NLP applications (and in this notebook), the "items" we care about are **words**.

### Why are they important?
When we process text, we need a way to turn words into numbers so that a machine learning model can understand them. Basic techniques treat text as an unordered bag of words. N-grams help us capture **local context** and **word order** by grouping words together.

### Feature-Based N-grams vs. Language-Model N-grams
- **Feature-based N-grams:** We use N-grams to create features (columns) in a dataset. For example, "machine learning" becomes a specific feature with a count, used for classification tasks.
- **Language-Model N-grams:** We use probabilities of N-grams to predict the next word in a sequence. If we see "machine", what is the probability the next word is "learning"?

### Common Applications
- **Text Classification** (e.g., categorizing news articles)
- **Spam Detection** (e.g., detecting phrases like "click here" or "free money")
- **Sentiment Analysis** (e.g., "not good" vs "good")
- **Search Engines** (understanding multi-word search queries)
- **Autocomplete & Next Word Prediction** (like your phone's keyboard)
- **Machine Translation & Speech Recognition**


---
## 2. What is an N-gram?

An N-gram is simply a sequence of N consecutive words extracted from a sentence. The "N" stands for a number.

Let's use a simple sentence: **"The cat sat on the mat"**

Here is how we break it down into different N-grams:

| N | Name | Definition | Examples from our sentence |
|---|---|---|---|
| 1 | **Unigram** | Single words | "The", "cat", "sat", "on", "the", "mat" |
| 2 | **Bigram** | Pairs of consecutive words | "The cat", "cat sat", "sat on", "on the", "the mat" |
| 3 | **Trigram** | Three consecutive words | "The cat sat", "cat sat on", "sat on the", "on the mat" |
| 4 | **4-gram** | Four consecutive words | "The cat sat on", "cat sat on the", "sat on the mat" |

As you can see, the window of size N slides across the sentence one word at a time.


---
## 3. Why do we need N-grams?

Before N-grams, the most common way to represent text was the **Bag-of-Words (BoW)** model.
Bag-of-Words simply counts how many times each word appears in a sentence, completely ignoring the order.

### The Limitation of Bag-of-Words
Consider these two sentences:
1. "I love machine learning"
2. "Machine learning loves me" *(Let's assume "love" and "loves" are treated as the same root word)*

In a pure Unigram (Bag-of-Words) model, both sentences have almost the identical representation. The model doesn't know *who* loves *what*. It just sees the words `['I', 'love', 'machine', 'learning']`.

### The N-gram Solution
If we use **Bigrams**, the models see:
1. "I love", "love machine", "machine learning"
2. "Machine learning", "learning loves", "loves me"

Now the representations are vastly different! N-grams **preserve local context** and capture short phrases, which can drastically change the meaning (e.g., "good" vs "not good").


---
## 4. Understanding CountVectorizer with N-grams

In Python, the `scikit-learn` library provides a powerful tool called `CountVectorizer` to easily convert text into a matrix of N-gram counts.

### Key Parameters:
- **`analyzer`**: Whether the feature should be made of word or character n-grams. (Default: 'word')
- **`lowercase`**: Converts all characters to lowercase before tokenizing. (Default: True)
- **`token_pattern`**: A regular expression indicating what constitutes a "token" (word).
- **`ngram_range`**: A tuple `(min_n, max_n)`. `(1, 1)` means only unigrams. `(1, 2)` means unigrams and bigrams.
- **`vocabulary`**: A dictionary mapping words to feature indices. (Can be learned automatically).

### Important Methods:
- **`fit()`**: Learns the vocabulary dictionary of all tokens in the raw documents.
- **`transform()`**: Transforms documents to a document-term matrix.
- **`fit_transform()`**: Does both `fit` and `transform` in a single step (more efficient).

### The Internal Pipeline
When you pass a sentence into `CountVectorizer`, here is what happens internally:

```text
       [Sentence] "The cat sat on the mat"
             ↓
     [Tokenization] Split into words (lowercased): ['the', 'cat', 'sat', 'on', 'the', 'mat']
             ↓
  [Generate N-grams] e.g., Bigrams: ['the cat', 'cat sat', ...]
             ↓
  [Build Vocabulary] Assign an ID to each unique N-gram: {'the cat': 0, 'cat sat': 1, ...}
             ↓
[Assign Feature Index] Map N-grams to matrix columns
             ↓
  [Count Frequencies] How many times did 'the cat' appear?
             ↓
    [Sparse Matrix] Store only non-zero counts to save memory
             ↓
 [Dense Matrix (opt)] Convert to a standard 2D array for easy viewing
```


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

# Create a small dataset of sentences (a "corpus")
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the rug.",
    "The cat chased the dog.",
    "Dogs and cats are great.",
    "The mat is on the rug."
]

print("Our Input Corpus:")
for i, sentence in enumerate(corpus):
    print(f"Sentence {i}: {sentence}")


In [ ]:
# 1. Initialize CountVectorizer for Unigrams (1, 1)
vectorizer_uni = CountVectorizer(ngram_range=(1, 1))

# 2. Fit and transform the corpus
X_uni = vectorizer_uni.fit_transform(corpus)

# 3. Get the learned vocabulary (feature names)
feature_names_uni = vectorizer_uni.get_feature_names_out()

# 4. Convert the Sparse Matrix to a Dense Matrix for viewing
dense_matrix_uni = X_uni.toarray()

# 5. Display as a Pandas DataFrame for clarity
df_uni = pd.DataFrame(dense_matrix_uni, columns=feature_names_uni)
df_uni.index = [f"Sentence {i}" for i in range(len(corpus))]

print(f"Vocabulary Size: {len(feature_names_uni)}")
display(df_uni)


### What happened?
We configured `CountVectorizer` to `ngram_range=(1, 1)`, which extracts only **Unigrams** (individual words).
The `fit_transform` method learned all unique words, sorted them alphabetically, and assigned them to columns.

### What should you observe?
- Punctuation was automatically removed.
- Everything was converted to lowercase (e.g., "The" -> "the").
- The table shows the **frequency count** of each word. For example, in Sentence 0, "the" appears twice, so its value is 2.
- Word order is completely lost. We just know which words are present.


---
## 6. Practical Example: Bigram Features

Now, let's see what happens if we change the `ngram_range` to `(2, 2)`, meaning we *only* want pairs of adjacent words.


In [ ]:
# 1. Initialize CountVectorizer for Bigrams (2, 2)
vectorizer_bi = CountVectorizer(ngram_range=(2, 2))

# 2. Fit and transform
X_bi = vectorizer_bi.fit_transform(corpus)

# 3. Get feature names and create DataFrame
feature_names_bi = vectorizer_bi.get_feature_names_out()
df_bi = pd.DataFrame(X_bi.toarray(), columns=feature_names_bi)
df_bi.index = [f"Sentence {i}" for i in range(len(corpus))]

print(f"Vocabulary Size: {len(feature_names_bi)}")
display(df_bi)


### What happened?
The features are now pairs of words (e.g., "cat chased", "cat sat").

### What should you observe?
- **Word order is preserved locally:** "cat sat" is a specific feature, distinct from "sat cat" (if it existed).
- **Context improves:** We now capture phrases like "on the" or "is on".
- **Vocabulary grows:** There are usually more unique bigrams than unigrams in a dataset, which is why the vocabulary size increased.
- **Sparsity increases:** Notice how many more `0`s (zeros) there are in this table compared to the unigram table.


---
## 7. Practical Example: Trigram Features

Let's expand the window to three words: `ngram_range=(3, 3)`.


In [ ]:
# Initialize CountVectorizer for Trigrams (3, 3)
vectorizer_tri = CountVectorizer(ngram_range=(3, 3))
X_tri = vectorizer_tri.fit_transform(corpus)

feature_names_tri = vectorizer_tri.get_feature_names_out()
df_tri = pd.DataFrame(X_tri.toarray(), columns=feature_names_tri)
df_tri.index = [f"Sentence {i}" for i in range(len(corpus))]

print(f"Vocabulary Size: {len(feature_names_tri)}")
display(df_tri)


### What happened?
Features are now triplets (e.g., "the cat sat").

### What should you observe?
- **Feature Explosion & Sparsity:** The matrix is mostly zeros. Each sentence has very specific trigrams that rarely overlap with other sentences.
- **When are trigrams useful?** They are great for capturing very specific phrases, names, or idioms (e.g., "New York City", "Natural Language Processing"), but they require a massive amount of text to be useful without being too sparse.


---
## 8. Mixed N-gram Features

In practice, using *only* bigrams or *only* trigrams can discard useful single words. Therefore, NLP practitioners often combine them!

Let's use `ngram_range=(1, 2)`. This means "extract unigrams AND bigrams".


In [ ]:
# Mix Unigrams and Bigrams (1, 2)
vectorizer_mixed = CountVectorizer(ngram_range=(1, 2))
X_mixed = vectorizer_mixed.fit_transform(corpus)

feature_names_mixed = vectorizer_mixed.get_feature_names_out()
df_mixed = pd.DataFrame(X_mixed.toarray(), columns=feature_names_mixed)
df_mixed.index = [f"Sentence {i}" for i in range(len(corpus))]

print(f"Vocabulary Size: {len(feature_names_mixed)}")
display(df_mixed.head(3)) # Displaying only first 3 sentences to save space


### Why combine them?
Combining them gives the machine learning model the "best of both worlds".
It knows that the document contains the word "good" (unigram), but it also knows if the document contains the phrase "not good" (bigram), allowing the model to interpret sentiment much more accurately!


---
## 9. N-gram Language Model for Next Word Prediction

So far, we used N-grams as **features** for a dataset (counting occurrences to build a matrix).
Now, let's switch gears and build an **N-gram Language Model**.

A Language Model tries to understand the probability of a sequence of words. We can use this to **predict the next word**.

### The Logic (Bigram Model)
If we want to predict the next word given the current word, we can look at our training text, find all the times the current word appeared, and see which words followed it most frequently!

Let's train a simple model using Python dictionaries (no external machine learning libraries!).


In [ ]:
from collections import defaultdict, Counter

# Small training corpus
train_corpus = [
    "the cat sat on the mat",
    "the cat ate fish",
    "the dog sat outside",
    "the dog chased the cat",
    "the cat likes milk"
]

# 1. Build the Bigram Language Model
bigram_model = defaultdict(Counter)

for sentence in train_corpus:
    # Tokenize: split sentence into words
    words = sentence.split()

    # 2. Generate bigrams and count frequencies
    for i in range(len(words) - 1):
        current_word = words[i]
        next_word = words[i+1]
        # 3. Store the count of the next word given the current word
        bigram_model[current_word][next_word] += 1

print("Trained Bigram Model Counts:")
for word, next_words in bigram_model.items():
    print(f"'{word}' is followed by: {dict(next_words)}")

print("-" * 40)

# 4. Function to predict the most likely next word
def predict_next_word_bigram(model, input_word):
    input_word = input_word.lower()
    if input_word in model:
        # Get the most common next word
        most_common = model[input_word].most_common(1)[0]
        return most_common[0]
    else:
        return "Unknown word"

# 5. Let's test it!
test_word_1 = "the"
print(f"Input: '{test_word_1}' -> Predicted next word: '{predict_next_word_bigram(bigram_model, test_word_1)}'")

test_word_2 = "cat"
print(f"Input: '{test_word_2}' -> Predicted next word: '{predict_next_word_bigram(bigram_model, test_word_2)}'")


### Extending to a Trigram Model
A bigram model only looks at ONE previous word to predict the next one.
A **trigram model** looks at TWO previous words to predict the next one. This provides much more context!


In [ ]:
# 1. Build the Trigram Language Model
trigram_model = defaultdict(Counter)

for sentence in train_corpus:
    words = sentence.split()
    # Need at least 3 words to make a trigram
    for i in range(len(words) - 2):
        # The context is now the previous TWO words
        context = (words[i], words[i+1])
        next_word = words[i+2]

        trigram_model[context][next_word] += 1

# 2. Function to predict next word using Trigram
def predict_next_word_trigram(model, word1, word2):
    context = (word1.lower(), word2.lower())
    if context in model:
        return model[context].most_common(1)[0][0]
    else:
        return "Unknown context"

# 3. Let's test it!
w1, w2 = "the", "cat"
print(f"Input: '{w1} {w2}' -> Predicted next word: '{predict_next_word_trigram(trigram_model, w1, w2)}'")

w1, w2 = "the", "dog"
print(f"Input: '{w1} {w2}' -> Predicted next word: '{predict_next_word_trigram(trigram_model, w1, w2)}'")


### How it works:
1. We slide a window over the text to group words (bigrams or trigrams).
2. We count how often Word B follows Word A (or how often Word C follows Word A and Word B).
3. To predict, we look up the history (the input word(s)) in our dictionary, and output the word that followed it most frequently during training.


---
## 10. Visual Comparison

Let's visualize how the choice of N-gram affects the **Vocabulary Size** and **Sparsity** (the percentage of zeroes in our matrix).


In [ ]:
# Calculate metrics for different N-gram ranges
ranges = [(1,1), (2,2), (3,3), (1,3)]
labels = ['Unigram', 'Bigram', 'Trigram', 'Mixed (1 to 3)']

vocab_sizes = []
sparsities = []

for r in ranges:
    vec = CountVectorizer(ngram_range=r)
    matrix = vec.fit_transform(corpus)

    # Vocab size is the number of columns
    vocab_sizes.append(matrix.shape[1])

    # Calculate sparsity: (number of zero elements) / (total elements)
    dense = matrix.toarray()
    zeroes = np.sum(dense == 0)
    total = dense.size
    sparsity_pct = (zeroes / total) * 100
    sparsities.append(sparsity_pct)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Vocabulary Size
axes[0].bar(labels, vocab_sizes, color=['skyblue', 'lightgreen', 'salmon', 'purple'])
axes[0].set_title('Vocabulary Size by N-gram Type')
axes[0].set_ylabel('Number of Features (Columns)')
axes[0].set_xlabel('N-gram Model')

# Plot 2: Matrix Sparsity
axes[1].bar(labels, sparsities, color=['skyblue', 'lightgreen', 'salmon', 'purple'])
axes[1].set_title('Matrix Sparsity by N-gram Type')
axes[1].set_ylabel('Percentage of Zeros (%)')
axes[1].set_xlabel('N-gram Model')

plt.tight_layout()
plt.show()


### What should you observe?
- **Vocabulary Size:** Moving from Unigrams to Trigrams, and especially mixing them, causes the vocabulary to grow rapidly. In large real-world text datasets, this growth is massive.
- **Sparsity:** As N increases, the matrices become extremely sparse. The Trigram matrix is almost entirely zeroes because specific 3-word phrases rarely repeat across different documents.


---
## 11. Advantages of N-grams

1. **Captures Local Context:** Unlike Bag-of-Words, bigrams can capture modifiers (e.g., "very good" vs "not good").
2. **Simple and Intuitive:** Easy to understand and implement without complex mathematics.
3. **Fast to Compute:** Counting words is computationally cheap compared to deep learning models.
4. **Strong Baselines:** Often used as a robust baseline for many NLP tasks before trying more complex models.


---
## 12. Limitations of N-grams

1. **Curse of Dimensionality / Vocabulary Explosion:** Adding bigrams and trigrams dramatically increases the number of features. For a vocabulary of 10,000 words, there are theoretically 100,000,000 possible bigrams!
2. **Extreme Sparsity:** Because most N-grams rarely appear, memory is wasted storing millions of zeroes.
3. **No Long-Range Dependencies:** A trigram model only looks back 2 words. It cannot connect the subject at the beginning of a long sentence to the verb at the end.
4. **No Semantic Understanding:** The model treats "car" and "automobile" as completely unrelated distinct features. It doesn't understand synonyms.


---
## 13. Mini Exercises

Time to test your knowledge! Try writing code to solve the following in new cells below:

1. **Custom N-grams:** Use `CountVectorizer` to extract bigrams from a new sentence of your choice.
2. **Vocabulary Counting:** Using a corpus of 3 distinct sentences, find out how much larger the `(1,3)` vocabulary is compared to just `(1,1)`.
3. **Matrix Comparison:** Print out the dense matrix for unigrams and bigrams for the sentence `"NLP is fun and NLP is cool"`. Observe the counts!
4. **Language Model Expansion:** Modify the Language Model dictionary code in Section 9 to predict the top **two** most likely next words, instead of just the number one word. *(Hint: look at the `.most_common(2)` method).*


---
## 14. Summary

Excellent work! Let's review what we learned:

| Concept | Key Takeaway |
|---|---|
| **N-gram** | A sequence of N consecutive words used to represent text. |
| **Unigram vs Bigram** | Unigrams are single words (lose order). Bigrams are word pairs (preserve local order). |
| **Bag-of-Words Limitation** | Bag-of-words ignores context. N-grams solve this by chaining words. |
| **CountVectorizer** | A `scikit-learn` tool that tokenizes text, builds a vocabulary of N-grams, and counts them. |
| **N-gram Language Model** | Uses the frequency of previous words (context) to predict the most likely next word. |
| **The Trade-off** | Higher N-grams improve context but cause vocabulary explosion and extreme matrix sparsity. |

N-grams are foundational to NLP. Modern Large Language Models (like ChatGPT) are incredibly advanced, but at their core, they are still fundamentally trying to perform the same task we did in Section 9: **Look at the previous words, and predict the next one!**
